# Calcul des performances
- Évaluation  

## Importations
- codecs pour les encodages
- pandas et numpy pour les calculs sur tableaux
- matplotlib pour les graphiques
- itertools pour les itérateurs sophistiqués (paires sur liste, ...)

In [82]:
# -*- coding: utf8 -*-
import codecs,operator,datetime,os,glob
import features
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools as it
import pickle
import networkx as nx
#%pylab inline
#pd.options.display.mpl_style = 'default'
debug=False
from __future__ import print_function

def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [83]:
%matplotlib inline

In [84]:
import yaml

In [85]:
from IPython.display import display, HTML

In [86]:
import datetime
def dateheure():
    return datetime.datetime.utcnow().strftime('%y%m%d%H%M')

In [87]:
saut="\n"

In [88]:
features.add_config('/Users/gilles/Github/SWIM/ParadigmGeneration/Vlexique2/bdlexique.ini')
fs=features.FeatureSystem('phonemes')

# Choix de l'échantillon et du gold
- *sampleFile* est le nom de l'échantillon de départ
- *goldFile* est le nom du lexique Gold de référence

In [89]:
repFiles="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"

num=4
inputFile="vlexique2-S%d.csv"%num
outputFile="vlexique2-S%d-omp-Swim2.csv"%num
goldFile="vlexique2-R%d.csv"%num

#num=4
#known="Train"
#predict="Test"
#inputFile="vlexique2-CV5-%s%d.csv"%(known,num)
#outputFile="vlexique2-CV5-%s%d-omp-Swim2.csv"%(known,num)
#goldFile="vlexique2-CV5-%s%d.csv"%(predict,num)

platinumFile="vlexique2-Total.csv"

fInput=repFiles+inputFile
fOutput=repFiles+outputFile
fGold=repFiles+goldFile
fPlatinum=repFiles+platinumFile

In [90]:
phonologicalMap="-X"
if "omp" in outputFile:
    casesType="-Morphomes"
else:
    casesType=""
listeFormesOutput=["FS","FP"]

### Dédoubler les lignes avec des surabondances dans *colonne*
>identifier une ligne avec surabondance

>>ajouter les lignes correspondant à chaque valeur

>>ajouter le numéro de la ligne initiale dans les lignes à supprimer

>supprimer les lignes avec surabondance

NB : il faut préparer le tableau pour avoir une indexation qui permette l'ajout des valeurs individuelles et la suppression des lignes de surabondances

In [91]:
def splitCellMates(df,colonne):
    '''
    Calcul d'une dataframe sans surabondance par dédoublement des valeurs
    '''
    test=df.reset_index()
    del test["index"]
    splitIndexes=[]
    for index,ligne in test.iterrows():
        if "," in ligne[colonne]:
            valeurs=set(ligne[colonne].split(","))
            nouvelleLigne=ligne
            for valeur in valeurs:
                nouvelleLigne[colonne]=valeur
                test=test.append(nouvelleLigne,ignore_index=True)
            splitIndexes.append(index)
    if splitIndexes:
        test=test.drop(test.index[splitIndexes])
    return test


# Lecture de l'échantillon

In [92]:
neutralisationsNORD=(u"6û",u"9ê")
neutralisationsSUD=(u"e2o",u"E9O")
if phonologicalMap=="-N":
    neutralisations=neutralisationsNORD
elif phonologicalMap=="-S":
    neutralisations=neutralisationsSUD
else:
    neutralisations=(u"",u"")
    phonologicalMap=("-X")
bdlexiqueIn = u"èò"+neutralisations[0]
bdlexiqueNum = [ord(char) for char in bdlexiqueIn]
neutreOut = u"EO"+neutralisations[1]
neutralise = dict(zip(bdlexiqueNum, neutreOut))

neutralisationsTotales=(u"e2o6û",u"E9O9ê")
totalNeutreIn=u"èò"+neutralisationsTotales[0]
totalNeutreNum=[ord(char) for char in totalNeutreIn]
totalNeutreOut=u"EO"+neutralisationsTotales[1]
totalNeutralise = dict(zip(totalNeutreNum, totalNeutreOut))

In [93]:
def recoder(chaine,table=neutralise):
    if type(chaine)==str:
        temp=chaine.translate(table)
        result=temp
    elif type(chaine)==unicode:
        result=chaine.translate(table)
    else:
        result=chaine
    return result

In [94]:
recoder("zj2t",totalNeutralise)

'zj9t'

### Vérification de la phonotactique des glides du français
- si *prononciation* est *None* renvoyer *None*
- ajout de diérèses dans les séquences mal-formées
- vérification des séquences consonne+glide à la finale

In [95]:
dierese={"j":"ij", "w":"uw","H":"yH","i":"ij","u":"uw","y":"yH"}
glide2voc={"j":"i","w":"u","H":"y"}

In [96]:
def checkFrench(prononciation):
    if prononciation and not pd.isnull(prononciation):
        result=recoder(prononciation)
        # Consonne plus glide final
        m=re.match(r"^(.*[^ieèEaOouy926êôâ])([jwH])$",result)
        if m:
            print ("pb avec un glide final", [prononciation])
            result=m.group(1)+glide2voc[m.group(2)]
        # attaque Obs+Liq+Glide
        m=re.match(r"(.*[ptkbdgfsSvzZ][rl])([jwH])(.*)",result)
        if m:
            n=re.search(r"[ptkbdgfsSvzZ][rl](wa|Hi|wê)",result)
            if not n:
                glide=m.group(2)
                result=m.group(1)+dierese[glide]+m.group(3)
        # Voyelle haute+Voyelle => diérèse
        m=re.match(r"(.*)([iuy])([ieEaOouy].*)",result)
        if m:
            glide=m.group(2)
            result=m.group(1)+dierese[glide]+m.group(3)
        # yod ou n palatal+yod
        m=re.match(r"(.*)([jJ])(j)(.*)",result)
        if m:
            result=m.group(1)+m.group(2)+m.group(4)
            print(prononciation,"=>",result)
        m=re.match(r"^(.*[^ieèEaOouy926êôâ])([jwH])6(.*)$",result)
        if m:
            result=m.group(1)+glide2voc[m.group(2)]+m.group(3)
            print(prononciation,"=>",result)
    else:
        result=prononciation
    return result

In [97]:
checkFrench("dEsj6ra")

dEsj6ra => dEsira


'dEsira'

In [98]:
echantillon=pd.read_csv(fInput,sep=";",encoding="utf8")
if u"Unnamed: 0" in echantillon.columns:
    del echantillon[u"Unnamed: 0"]
echantillon=echantillon.dropna(axis=1,how='all')
print(len(echantillon.columns))
echantillon.head()

52


,lexeme,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1P,fi1S,fi2P,...,ppFP,ppFS,ppMP,ppMS,ps1P,ps1S,ps2P,ps2S,ps3P,ps3S
0,abaisser,NaN,NaN,NaN,NaN,NaN,NaN,NaN,abEs6rE,NaN,...,abEse,abEse,abEse,abEse,NaN,abEs,NaN,abEs,NaN,abEs
1,abandonner,NaN,abâdOnE,NaN,NaN,abâdOnEr,abâdOna,abâdOn6rô,abâdOn6rE,abâdOn6re,...,abâdOne,abâdOne,abâdOne,abâdOne,abâdOnjô,abâdOn,abâdOnje,abâdOn,abâdOn,abâdOn
2,abasourdir,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,abazurdi,abazurdi,abazurdi,NaN,NaN,NaN,NaN,NaN,NaN
3,abattre,NaN,NaN,NaN,NaN,abatir,abati,abatrô,abatrE,abatre,...,abaty,abaty,abaty,abaty,NaN,abat,abatje,abat,abat,abat
4,abdiquer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,abdike,NaN,NaN,NaN,NaN,NaN,abdik


In [99]:
paradigmes=pd.read_csv(fOutput,sep=";",encoding="utf8")
if u"Unnamed: 0" in paradigmes.columns:
    del paradigmes[u"Unnamed: 0"]
paradigmes=paradigmes.dropna(axis=1,how='all')
print((paradigmes.columns))
paradigmes.head()

Index(['ai1P', 'ai1S', 'ai2P', 'ai2S', 'ai3P', 'ai3S', 'fi1P', 'fi1S', 'fi2P',
       'fi2S', 'fi3P', 'fi3S', 'ii1P', 'ii1S', 'ii2P', 'ii2S', 'ii3P', 'ii3S',
       'inf', 'is1P', 'is1S', 'is2P', 'is2S', 'is3P', 'is3S', 'lexeme', 'pI1P',
       'pI2P', 'pI2S', 'pP', 'pc1P', 'pc1S', 'pc2P', 'pc2S', 'pc3P', 'pc3S',
       'pi1P', 'pi1S', 'pi2P', 'pi2S', 'pi3P', 'pi3S', 'ppFP', 'ppFS', 'ppMP',
       'ppMS', 'ps1P', 'ps1S', 'ps2P', 'ps2S', 'ps3P', 'ps3S'],
      dtype='object')


,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1P,fi1S,fi2P,fi2S,...,ppFP,ppFS,ppMP,ppMS,ps1P,ps1S,ps2P,ps2S,ps3P,ps3S
0,abEsam,abEsE,NaN,abEsa,abEsEr,abEsa,abEs6rô,abEs6rE,abEs6re,abEs6ra,...,abEse,abEse,abEse,abEse,NaN,abEs,abEsje,abEs,abEs,abEs
1,abâdOnam,abâdOnE,NaN,abâdOna,abâdOnEr,abâdOna,abâdOn6rô,abâdOn6rE,abâdOn6re,abâdOn6ra,...,abâdOne,abâdOne,abâdOne,abâdOne,abâdOnjô,abâdOn,abâdOnje,abâdOn,abâdOn,abâdOn
2,NaN,abazurdi,NaN,abazurdi,abazurdir,abazurdi,abazurdirô,abazurdirE,abazurdire,abazurdira,...,abazurdi,abazurdi,abazurdi,abazurdi,NaN,abazurdis,abazurdisje,abazurdis,abazurdis,abazurdis
3,NaN,NaN,NaN,abati,abatir,abati,abatrô,abatrE,abatre,abatra,...,abaty,abaty,abaty,abaty,abatjô,abat,abatje,abat,abat,abat
4,abdikam,abdikE,NaN,abdika,abdikEr,abdika,abdik6rô,abdik6rE,abdik6re,abdik6ra,...,abdike,abdike,abdike,abdike,abdikjô,abdik,abdikje,abdik,abdik,abdik


In [100]:
sampleCases=paradigmes.columns.values.tolist()
sampleCases.remove(u"lexeme")
# sampleCases
analyseCases=sampleCases

#Adapt all the forms to French phonology
for case in sampleCases:
    paradigmes[case]=paradigmes[case].apply(lambda x: checkFrench(x))

aliJjô => aliJô
kliJjô => kliJô
kOJjô => kOJô
kôsiJjô => kôsiJô
grOJjô => grOJô
lOrJjô => lOrJô
maJjô => maJô
râfrOJjô => râfrOJô
rOJjô => rOJô
realiJjô => realiJô
rEpyJjô => rEpyJô
suliJjô => suliJô
syrliJjô => syrliJô
trEpiJjô => trEpiJô
EbOrJjô => EbOrJô
aliJje => aliJe
kliJje => kliJe
kOJje => kOJe
kôsiJje => kôsiJe
grOJje => grOJe
lOrJje => lOrJe
maJje => maJe
râfrOJje => râfrOJe
rOJje => rOJe
realiJje => realiJe
rEpyJje => rEpyJe
suliJje => suliJe
syrliJje => syrliJe
trEpiJje => trEpiJe
EbOrJje => EbOrJe
aliJjô => aliJô
kliJjô => kliJô
kOJjô => kOJô
kôsiJjô => kôsiJô
grOJjô => grOJô
lOrJjô => lOrJô
maJjô => maJô
râfrOJjô => râfrOJô
rOJjô => rOJô
realiJjô => realiJô
rEpyJjô => rEpyJô
suliJjô => suliJô
syrliJjô => syrliJô
trEpiJjô => trEpiJô
EbOrJjô => EbOrJô
aZ6nujje => aZ6nuje
aliJje => aliJe
bafujje => bafuje
barbujje => barbuje
bidujje => biduje
br6dujje => br6duje
brujje => bruje
kafujje => kafuje
Satujje => Satuje
kliJje => kliJe
kOJje => kOJe
kôsiJje => kôsiJe
dujje => duje


- sampleCases pour la liste des cases effectivement représentées dans le corpus de départ 

In [101]:
countInput=echantillon[sampleCases].stack().value_counts(dropna=True).sum()
print("nombre de formes de départ",countInput)

nombre de formes de départ 53485


In [102]:
gold=pd.read_csv(fGold,sep=";",encoding="utf8")
if u"Unnamed: 0" in gold.columns:
    del gold[u"Unnamed: 0"]
gold=gold.dropna(axis=1,how='all')

for case in sampleCases:
    gold[case]=gold[case].apply(lambda x: checkFrench(x))

gold.head()

,lexeme,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1P,fi1S,fi2P,...,ppFP,ppFS,ppMP,ppMS,ps1P,ps1S,ps2P,ps2S,ps3P,ps3S
0,abaisser,NaN,abEsE,abEsat,NaN,abEsEr,abEsa,abEs6rô,NaN,abEs6re,...,NaN,NaN,NaN,NaN,abEsjô,NaN,NaN,NaN,abEs,NaN
1,abandonner,abâdOnam,NaN,abâdOnat,abâdOna,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,abasourdir,NaN,NaN,NaN,abazurdi,NaN,NaN,NaN,abazurdirE,NaN,...,abazurdi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,abazurdis
3,abattre,NaN,NaN,NaN,abati,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,abatjô,NaN,NaN,NaN,NaN,NaN
4,abdiquer,NaN,NaN,NaN,NaN,NaN,abdika,abdik6rô,abdik6rE,abdik6re,...,NaN,NaN,abdike,NaN,NaN,abdik,NaN,abdik,NaN,NaN


In [103]:
countPredictions=gold[sampleCases].stack().value_counts(dropna=True).sum()
print("nombre de formes à générer",countPredictions)

nombre de formes à générer 55563


In [104]:
platinum=pd.read_csv(fPlatinum,sep=";",encoding="utf8")
if u"Unnamed: 0" in gold.columns:
    del platinum[u"Unnamed: 0"]
platinum=platinum.dropna(axis=1,how='all')

for case in sampleCases:
    platinum[case]=platinum[case].apply(lambda x: checkFrench(x))

platinum.head()

fajisisjô,fajjô => fajisisjô,fajô
fajisisje,fajje => fajisisje,faje


,lexeme,ai1P,ai1S,ai2P,ai2S,ai3P,ai3S,fi1P,fi1S,fi2P,...,ppFP,ppFS,ppMP,ppMS,ps1P,ps1S,ps2P,ps2S,ps3P,ps3S
0,abaisser,abEsam,abEsE,abEsat,abEsa,abEsEr,abEsa,abEs6rô,abEs6rE,abEs6re,...,abEse,abEse,abEse,abEse,abEsjô,abEs,abEsje,abEs,abEs,abEs
1,abandonner,abâdOnam,abâdOnE,abâdOnat,abâdOna,abâdOnEr,abâdOna,abâdOn6rô,abâdOn6rE,abâdOn6re,...,abâdOne,abâdOne,abâdOne,abâdOne,abâdOnjô,abâdOn,abâdOnje,abâdOn,abâdOn,abâdOn
2,abasourdir,abazurdim,abazurdi,abazurdit,abazurdi,abazurdir,abazurdi,abazurdirô,abazurdirE,abazurdire,...,abazurdi,abazurdi,abazurdi,abazurdi,abazurdisjô,abazurdis,abazurdisje,abazurdis,abazurdis,abazurdis
3,abattre,abatim,abati,abatit,abati,abatir,abati,abatrô,abatrE,abatre,...,abaty,abaty,abaty,abaty,abatjô,abat,abatje,abat,abat,abat
4,abdiquer,abdikam,abdikE,abdikat,abdika,abdikEr,abdika,abdik6rô,abdik6rE,abdik6re,...,abdike,abdike,abdike,abdike,abdikjô,abdik,abdikje,abdik,abdik,abdik


In [105]:
countLexemes=len(paradigmes.dropna(thresh=1)["lexeme"])

# Calcul des performances

In [106]:
correct=0
correctNeut=0
different=0
differentNeut=0
missing=0
badLexemes=set()
badNeutLexemes=set()
for ix,row in echantillon.iloc[:,:].iterrows():
    # print(row.lexeme)
    dictPlatinum=row.dropna().to_dict()
    del dictPlatinum["lexeme"]
    # print("Echantillon",dictPlatinum)
    selPlatinum=[]
    selResultats=[]
    for k,v in dictPlatinum.items():
        selPlatinum.append("(platinum.%s=='%s')"%(k,v))
        if k in paradigmes.columns:
            selResultats.append("(paradigmes.%s=='%s')"%(k,v))
    exec("%s=%s"%("testPlatinum","&".join(selPlatinum)))
    if selResultats:
        exec("%s=%s"%("testResultats","&".join(selResultats)))
    else:
        testResultats=[]
    lCandidats=platinum.loc[testPlatinum,:]["lexeme"].tolist()
    dictCandidats=gold.loc[gold.lexeme.isin(lCandidats),:].T.dropna().to_dict()
    # print(dictCandidats)
    if not isinstance(testResultats,list):
        lResultats=paradigmes.loc[testResultats,:]["lexeme"].tolist()
    else:
        lResultats=[row.lexeme]
    # print(lResultats)
    lCorrect=0
    lCorrectNeut=0
    lDifferent=0
    lDifferentNeut=0
    lMissing=0
    for ik,candidat in dictCandidats.items():
        for case,forme in candidat.items():
            for lexeme in lResultats:
                if case in paradigmes.columns and case!="lexeme":
                    sForme=paradigmes.loc[paradigmes.lexeme==lexeme,case]
                    # print(sForme)
                    if (sForme==forme).all():
                        # print("correct",case,end=", ")
                        lCorrect+=1
                    elif (sForme==sForme).all():                        
                        lDifferent+=1
                        badLexemes.add(lexeme)
                        # print(forme,sForme)
                        if recoder(forme,totalNeutralise)==recoder(sForme.values[0],totalNeutralise):
                            lCorrectNeut+=1
                            #print("variante",lexeme,case,forme,sForme.tolist())
                        else:
                            lDifferentNeut+=1
                            #print("different",lexeme,case,forme,sForme.tolist())
                            badNeutLexemes.add(lexeme)
                    else:
                        # print("missing",case,end=", ")
                        lMissing+=1
    # print()
    # print(row.lexeme,lCorrect,lDifferent,lMissing)
    correct+=lCorrect
    different+=lDifferent
    correctNeut+=lCorrectNeut
    differentNeut+=lDifferentNeut
    missing+=lMissing
    # print()
correct,different,missing,correctNeut,differentNeut

(47444, 143, 1318, 2, 141)

In [107]:
precision=float(correct)/(correct+different)*100
rappel=float(correct)/(correct+missing)*100
print("input",inputFile,"output",outputFile)
print("Brut précision %.1f, rappel %.1f"%(precision,rappel))
precisionNeut=float(correctNeut+correct)/(correct+correctNeut+differentNeut)*100
rappelNeut=float(correct+correctNeut)/(correct+correctNeut+missing)*100
print("Neutralisée précision %.1f, rappel %.1f"%(precisionNeut,rappelNeut))


input vlexique2-S4.csv output vlexique2-S4-omp-Swim2.csv
Brut précision 99.7, rappel 97.3
Neutralisée précision 99.7, rappel 97.3


In [108]:
print(len(badLexemes),"\n"+", ".join(badLexemes))

45 
prévaloir, répondre, expatrier, médire, monnayer, conseiller, recéler, déblayer, devoir, justifier, avoir, défrayer, nuire, suffire, rudoyer, convoyer, strier, déconseiller, pagayer, être, effrayer, gésir, traduire, vouvoyer, fuir, élire, apeurer, seoir, dévoyer, relire, étayer, circoncire, intervertir, escorter, dépourvoir, affermir, fauter, prévoir, bégayer, dédire, déchoir, revenir, frayer, guerroyer, receler
